# Reference-attention replay on the twelve-carrier transferred cache, Kaggle T4

Same Tesla T4, same Qwen3-4B revision, FP16 weights, greedy settings. Do not switch GPUs. Oracle diagnostic, not a method. No adapter.

Reference: **fk_fv** (fresh K + fresh V under the same generic support). Same compact layout as the transferred cache; the full-A cache is not used.

Branches, all teacher-forced on the same frozen twelve-line plan prompt plus the reference's saved plan text from the K/V swap run:
- **tk_tv**: transferred cache, ordinary attention
- **fk_tv**: fresh keys + transferred values
- **fk_fv**: reference
- **tk_tv_refattn**: transferred cache; every layer uses the reference row's attention weights on tk_tv's own values (two-row forward)
- **tk_tv_selfattn**: transferred cache replaying its own attention (numerics control)

Reported: KL from the reference distributions, argmax agreement, and label decisions at the 12 eligibility-label positions per case. **Not free-generation accuracy.**

**Cell 4** (`--phase check`): replaying a branch's own attention, and the batched instrumented forward without replacement, must reproduce ordinary predictions (mean KL <= 0.01, argmax agreement >= 0.98, identical label decisions); replacement must hit every layer. Stop if it fails.

**Cell 5** (`--phase compare`): adds the reference-attention intervention, one case at a time.

Upload `src/`, `configs/`, `scripts/`, `data/plan4_timing12.jsonl`; from `results_stage_plan4_kvmix12/`: `a_artifacts_12.jsonl`, `evaluations.jsonl`; from `results_stage_plan4_recon12/`: `check_summary.json`.

Download `/kaggle/working/results_stage_plan4_refattn12`.

In [ ]:
import os, json, shutil, sys
from pathlib import Path

os.environ["LATENT_RELAY_INFER_DTYPE"] = "fp16"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()

candidates = [Path.cwd()]
if Path("/kaggle/input").is_dir():
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d not in {".git", ".pytest_cache"}]
        if "stage_plan4_refattn12.py" in files and Path(root).name == "src":
            candidates.append(Path(root).parent)

REPO = None
for cand in candidates:
    if (cand / "src" / "stage_plan4_refattn12.py").is_file() and (cand / "data" / "plan4_timing12.jsonl").is_file():
        REPO = cand
        break
if REPO is None:
    raise FileNotFoundError("Need src/stage_plan4_refattn12.py and data/plan4_timing12.jsonl.")

if ON_KAGGLE:
    dest = WORK / "latent-relay-audit"
    if str(REPO).startswith("/kaggle/input") or not (dest / "src" / "stage_plan4_refattn12.py").is_file():
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    REPO = dest
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("REPO", REPO.resolve())
for folder, names in (("results_stage_plan4_kvmix12", ("a_artifacts_12.jsonl", "evaluations.jsonl")),
                      ("results_stage_plan4_recon12", ("check_summary.json",))):
    for name in names:
        print(folder, name, (REPO / folder / name).is_file())

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.3", "tokenizers==0.22.2", "huggingface_hub==0.36.2",
    "safetensors==0.7.0", "accelerate==1.13.0", "datasets==4.4.1",
])
import torch, transformers
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("cuda", torch.cuda.is_available(), gpu)
print("transformers", transformers.__version__)
assert transformers.__version__.startswith("4.57")
if gpu is not None and "T4" not in gpu:
    raise RuntimeError(f"Stay on the locked Tesla T4. Got {gpu}.")

In [ ]:
subprocess.check_call([sys.executable, "scripts/fetch_upstream.py"])
print("Dest /kaggle/working/results_stage_plan4_refattn12. Check first, then compare.")

In [ ]:
rc = subprocess.call([sys.executable, "-m", "src.stage_plan4_refattn12", "--split", "plan4_timing12", "--phase", "check"])
print("refattn12 check exit", rc)
path = WORK / "results_stage_plan4_refattn12" / "check_summary.json"
if not path.is_file():
    path = REPO / "results" / "stage_plan4_refattn12" / "run" / "check_summary.json"
check = json.loads(path.read_text(encoding="utf-8"))
print(json.dumps({"n": check.get("n"), "all_passed": check.get("all_passed"), "tolerance": check.get("tolerance")}, indent=2))
for row in check.get("rows", []):
    if not row["ok"]:
        print(row["example_id"], json.dumps(row["checks"], indent=1))
if not check.get("all_passed"):
    raise RuntimeError("Attention instrumentation does not reproduce ordinary predictions. Do not run compare.")

In [ ]:
rc = subprocess.call([sys.executable, "-m", "src.stage_plan4_refattn12", "--split", "plan4_timing12", "--phase", "compare"])
print("refattn12 compare exit", rc)

In [ ]:
src = WORK / "results_stage_plan4_refattn12"
if not (src / "summary.json").is_file():
    src = REPO / "results" / "stage_plan4_refattn12" / "run"
data = json.loads((src / "summary.json").read_text(encoding="utf-8"))
print("cases", data["n_cases"], "of", data["n_expected_cases"], "| instrumentation_ok", data["instrumentation_ok"])
keys = ("labels_correct", "n_labels", "labels_agree_ref", "ineligible_marked_eligible", "eligible_marked_ineligible",
        "cases_all_labels_correct", "mean_kl_vs_reference", "mean_label_kl_vs_reference", "mean_argmax_agree_reference")
for b, block in data["branches"].items():
    print(b, {k: block.get(k) for k in keys})
print("\nrecovery toward reference", data["recovery_toward_reference"])
for name, block in data["paired_by_case"].items():
    c = block["labels_correct"]
    print(name, {k: c.get(k) for k in ("total_delta", "ci95_per_case", "cases_increased", "cases_decreased")})

In [ ]:
# Export compare_checks.json from this session (no model rerun).
run_file = REPO / "results" / "stage_plan4_refattn12" / "run" / "compare_checks.json"
out_file = WORK / "results_stage_plan4_refattn12" / "compare_checks.json"
if run_file.is_file() and not out_file.is_file():
    out_file.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(run_file, out_file)
print("compare_checks.json", out_file.is_file(), out_file)
if out_file.is_file():
    cc = json.loads(out_file.read_text(encoding="utf-8"))
    print({"n": cc.get("n"), "all_passed": cc.get("all_passed")})